# Climate Data Fetch & Processing

Fetches downscaled CMIP6 climate data from Cal-Adapt via climakitae,
processes it (spatial masking, anomaly calculation), and saves incremental CSVs
for downstream R plotting.

**Variables**: T_Max, T_Min → T_Avg, Precipitation  
**Resolution**: 3 km Statistical downscaling  
**Scenarios**: Historical, SSP 2-4.5, SSP 3-7.0, SSP 5-8.5  
**Regions**: Joshua Tree NP, Mojave National Preserve

In [9]:
import climakitae as ck
from climakitae.core.data_interface import get_data
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray as rxr
import warnings
import os
import gc

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [10]:
# --- Configuration ---

STANDARD_CRS = "EPSG:4326"  # WGS84
DOWNSCALING = "Statistical"
RES = "3 km"

VARIABLES_STAT = {
    "T_Max": "Maximum air temperature at 2m",
    "T_Min": "Minimum air temperature at 2m",
    "Precip": "Precipitation (total)"
}

SCENARIOS = ["Historical Climate", "SSP 2-4.5", "SSP 3-7.0", "SSP 5-8.5"]

TIMESPAN = (1950, 2100)
BASELINEPER = (1995, 2014)
HISENDYEAR = 2014
SMOOTHING_WINDOW = 10

# Paths — relative to this notebook's location in notebooks/python/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
OUTPUT_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "csv")
PARK_OUTLINES_DIR = os.path.join(PROJECT_ROOT, "parkOutlines")

os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

output_filename = os.path.join(OUTPUT_DATA_DIR, "Statistical_3km_Tavg_Precip_Incremental_Resumable.csv")

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUTPUT_DATA_DIR}")
print(f"Shapefiles:   {PARK_OUTLINES_DIR}")

Project root: /Users/andrewschoenen/Desktop/DSE
Output dir:   /Users/andrewschoenen/Desktop/DSE/data/csv
Shapefiles:   /Users/andrewschoenen/Desktop/DSE/parkOutlines


In [11]:
# --- Load Park Boundaries ---

SHAPEFILES = {
    "JoshuaTree": os.path.join(PARK_OUTLINES_DIR, "JoshuaTree", "Joshua_Tree_National_Park.shp"),
    "Mojave": os.path.join(PARK_OUTLINES_DIR, "Mojave", "Mojave_National_Preserve.shp")
}

BOUNDARIES = {}
for name, path in SHAPEFILES.items():
    try:
        boundary_WGS84 = gpd.read_file(path)
        if boundary_WGS84.crs is None or str(boundary_WGS84.crs) != STANDARD_CRS:
            boundary_WGS84 = boundary_WGS84.to_crs(STANDARD_CRS)
        BOUNDARIES[name] = boundary_WGS84
        print(f"Loaded {name} boundary.")
    except Exception as e:
        print(f"ERROR: Couldn't load {name} bounding file: {e}")
        BOUNDARIES[name] = None

Loaded JoshuaTree boundary.
Loaded Mojave boundary.


In [12]:
# --- Helper Functions ---

def preprocess_data(dataArray, varKey):
    """Converts units and ensures CRS/Spatial Dims are set correctly."""
    if dataArray is None:
        return None

    # Unit conversions
    if 'T_' in varKey and dataArray.attrs.get('units') == 'K':
        dataArray = dataArray - 273.15
        dataArray.attrs['units'] = '°C'

    if varKey == 'Precip' and dataArray.attrs.get('units') in ['kg/m^2/s', 'kg m-2 s-1']:
        days_in_month = dataArray.time.dt.days_in_month
        seconds_per_day = 86400
        dataArray = (dataArray * seconds_per_day) * days_in_month
        dataArray.attrs['units'] = 'mm/month'

    # Ensure CRS is set
    if dataArray.rio.crs is None:
        try:
            dataArray = dataArray.rio.write_crs(STANDARD_CRS)
        except Exception:
            pass

    # Identify and set spatial dimensions
    y_dim = None
    x_dim = None

    if 'latitude' in dataArray.dims: y_dim = 'latitude'
    elif 'lat' in dataArray.dims: y_dim = 'lat'
    elif 'y' in dataArray.dims: y_dim = 'y'

    if 'longitude' in dataArray.dims: x_dim = 'longitude'
    elif 'lon' in dataArray.dims: x_dim = 'lon'
    elif 'x' in dataArray.dims: x_dim = 'x'

    if y_dim is None or x_dim is None:
        print(f"  Error: Could not identify spatial dimensions. Dims found: {dataArray.dims}")
        return None

    try:
        dataArray = dataArray.rio.set_spatial_dims(x_dim=x_dim, y_dim=y_dim)
    except Exception as e:
        print(f"  Error setting spatial dims: {e}")
        return None

    return dataArray

In [13]:
def mask_and_spatial_average(dataArray, boundaryGDF_WGS84):
    """Masks data to boundary and calculates area-weighted spatial average."""
    if dataArray is None:
        return None

    try:
        coordsToDrop = []
        x_dim = dataArray.rio.x_dim
        y_dim = dataArray.rio.y_dim

        if x_dim != 'lon' and 'lon' in dataArray.coords: coordsToDrop.append('lon')
        if y_dim != 'lat' and 'lat' in dataArray.coords: coordsToDrop.append('lat')

        dataCleaned = dataArray.drop_vars(coordsToDrop, errors='ignore')

        spatialDims = (y_dim, x_dim)
        nonSpatialDims = [dim for dim in dataCleaned.dims if dim not in spatialDims]
        expectedOrder = tuple(nonSpatialDims) + spatialDims
        if dataCleaned.dims != expectedOrder:
            dataCleaned = dataCleaned.transpose(*expectedOrder)

    except Exception as e:
        print(f"  Error preparing data structure: {e}")
        return None

    try:
        if str(dataCleaned.rio.crs) != str(boundaryGDF_WGS84.crs):
            boundaryNativeCRS = boundaryGDF_WGS84.to_crs(dataCleaned.rio.crs)
        else:
            boundaryNativeCRS = boundaryGDF_WGS84

        maskedData = dataCleaned.rio.clip(
            boundaryNativeCRS.geometry.values,
            drop=False,
            all_touched=True
        )
    except Exception as e:
        print(f"  Error during masking: {e}")
        return None

    try:
        latitudes = maskedData[y_dim]
        weights = np.cos(np.deg2rad(latitudes))
        weights.name = "weights"

        try:
            spatialAVG = maskedData.weighted(weights).mean(dim=[x_dim, y_dim], skipna=True)
        except Exception as weight_e:
            print(f"    Weighted average failed: {weight_e}. Using unweighted mean.")
            spatialAVG = maskedData.mean(dim=[x_dim, y_dim], skipna=True)

        spatialAVG.load()

        if spatialAVG.isnull().all():
            return None

        return spatialAVG

    except Exception as e:
        print(f"    Error in calculating spatial average or loading data: {e}.")
        return None

In [14]:
def calculate_smoothed_anomalies(spatialAVG, varKey, scenario):
    """Performs temporal aggregation, anomaly calculation, and smoothing."""
    if spatialAVG is None:
        return None

    # Temporal aggregation
    if varKey == 'Precip':
        annualData = spatialAVG.resample(time='YE').sum(dim='time', skipna=True)
        annualData.attrs['units'] = 'mm/year'
    else:
        annualData = spatialAVG.resample(time='YE').mean(dim='time', skipna=True)

    # Anomaly calculations
    baselineSlice = annualData.sel(time=slice(str(BASELINEPER[0]), str(BASELINEPER[1])))

    if baselineSlice.time.size == 0:
        print("    Warning: No data found for baseline period in this scenario segment.")
        return None

    baseline_mean = baselineSlice.mean(dim='time')

    if varKey == 'Precip':
        if (np.abs(baseline_mean) < 1.0).any():
            print("    Note: Baseline precip < 1mm/year detected. Using absolute difference (Δmm/year).")
            anomalies = (annualData - baseline_mean)
        else:
            anomalies = ((annualData - baseline_mean) / baseline_mean) * 100
    else:
        anomalies = (annualData - baseline_mean)

    # Smoothing
    smoothed_anomalies = anomalies.rolling(time=SMOOTHING_WINDOW, center=True, min_periods=1).mean()

    # Convert to DataFrame
    sdf = smoothed_anomalies.to_dataframe(name='Anomaly').reset_index()
    sdf['Variable'] = varKey
    sdf['Year'] = sdf['time'].dt.year

    sdf['DataScenario'] = scenario

    if scenario == "Historical Climate":
        sdf['Scenario'] = "Historical Climate"
    else:
        sdf['Scenario'] = np.where(
            sdf['Year'] <= HISENDYEAR,
            "Historical Climate",
            scenario
        )

    if 'simulation' in sdf.columns:
        sdf = sdf.rename(columns={'simulation': 'Simulation'})
    elif 'source_id' in sdf.columns:
        sdf = sdf.rename(columns={'source_id': 'Simulation'})
    else:
        if 'simulation' in spatialAVG.coords:
            sdf['Simulation'] = str(spatialAVG.simulation.values.item())
        elif 'source_id' in spatialAVG.coords:
            sdf['Simulation'] = str(spatialAVG.source_id.values.item())
        else:
            sdf['Simulation'] = "Unknown"

    return sdf

In [15]:
# --- Main Processing Loop (Resumable Incremental) ---
import time as _time

required_cols = ['Year', 'Region', 'Scenario', 'DataScenario', 'Simulation', 'Variable', 'Anomaly']

# --- Build the full work manifest ---
# Each unit of work = (region, scenario, variable)
# This is the finest granularity we can checkpoint at.
ALL_WORK = []
for region_name in BOUNDARIES:
    for scenario in SCENARIOS:
        for var_key in ["T_Avg", "Precip"]:
            ALL_WORK.append((region_name, scenario, var_key))

print(f"Total work units: {len(ALL_WORK)}  (2 regions x 4 scenarios x 2 output vars)")

# --- Load checkpoint from existing CSV ---
processed_units = set()

if os.path.exists(output_filename):
    try:
        existing_df = pd.read_csv(output_filename, low_memory=False)
        if {'Region', 'DataScenario', 'Variable'}.issubset(existing_df.columns):
            processed_units = set(zip(
                existing_df['Region'],
                existing_df['DataScenario'],
                existing_df['Variable']
            ))
            print(f"Resuming: {len(processed_units)}/{len(ALL_WORK)} units already done.")
        else:
            print("Existing CSV missing columns. Starting fresh.")
            pd.DataFrame(columns=required_cols).to_csv(output_filename, index=False)
    except pd.errors.EmptyDataError:
        print("Existing CSV empty. Starting fresh.")
        pd.DataFrame(columns=required_cols).to_csv(output_filename, index=False)
    except Exception as e:
        print(f"Error reading CSV: {e}. Starting fresh.")
        pd.DataFrame(columns=required_cols).to_csv(output_filename, index=False)
else:
    pd.DataFrame(columns=required_cols).to_csv(output_filename, index=False)
    print("No existing CSV. Starting fresh.")

remaining = [w for w in ALL_WORK if w not in processed_units]
print(f"Remaining: {len(remaining)} units to process")
print(f"Output: {output_filename}")
print()

Total work units: 16  (2 regions x 4 scenarios x 2 output vars)
Resuming: 0/16 units already done.
Remaining: 16 units to process
Output: /Users/andrewschoenen/Desktop/DSE/data/csv/Statistical_3km_Tavg_Precip_Incremental_Resumable.csv



In [ ]:
# --- Processing Loop (with parallel .load()) ---
# 
# IMPORTANT: get_data() is NOT thread-safe — it creates DataParameters objects
# that fetch boundary data from S3, causing race conditions in threads.
# 
# Strategy: Call get_data() SEQUENTIALLY to get lazy dask DataArrays (~29s each),
# then call .load() in PARALLEL threads to overlap the ~90s network I/O.
# Saves after EVERY (region, scenario, variable) for incremental persistence.

from concurrent.futures import ThreadPoolExecutor, as_completed
import time as _time

run_start = _time.perf_counter()
completed_this_run = 0
total_remaining = len(remaining)

def _load_one(var_key, scenario, lazy_da, boundary_gdf):
    """Load a lazy DataArray and run the full processing pipeline in a thread.
    Only .load() and numpy/xarray ops — no get_data() calls."""
    try:
        spatial_avg = mask_and_spatial_average(lazy_da, boundary_gdf)
        return (var_key, scenario, spatial_avg)
    except Exception as e:
        print(f"    LOAD ERROR: {var_key} | {scenario}: {e}")
        return (var_key, scenario, None)

# --- Process one region at a time ---
for region_name, boundary_gdf in BOUNDARIES.items():
    if boundary_gdf is None:
        continue
    
    bounds = boundary_gdf.total_bounds
    lat_bounds = (bounds[1], bounds[3])
    lon_bounds = (bounds[0], bounds[2])

    # Figure out which (scenario, var) combos still need processing
    needed_scenarios = set()
    for (rn, sc, vk) in remaining:
        if rn == region_name:
            needed_scenarios.add(sc)
    
    if not needed_scenarios:
        print(f"\n--- {region_name}: All done, skipping ---")
        continue

    print(f"\n{'=' * 60}")
    print(f"REGION: {region_name}")
    print(f"  Scenarios to fetch: {sorted(needed_scenarios)}")
    
    # ================================================================
    # PHASE 1: Serial get_data() to build lazy DataArrays
    #   get_data() is fast (~29s) — it's just metadata + catalog lookup.
    #   The expensive part (.load()) happens in Phase 2.
    # ================================================================
    
    fetch_jobs = []
    for scenario in needed_scenarios:
        for var_key, var_name in VARIABLES_STAT.items():
            fetch_jobs.append((var_key, var_name, scenario))
    
    print(f"  Phase 1: Fetching {len(fetch_jobs)} lazy arrays (serial, ~29s each)...")
    fetch_start = _time.perf_counter()
    
    # Key: scenario -> {var_key: preprocessed lazy DataArray}
    lazy_data = {}
    for var_key, var_name, scenario in fetch_jobs:
        t0 = _time.perf_counter()
        try:
            d = get_data(
                variable=var_name,
                resolution=RES,
                downscaling_method=DOWNSCALING,
                timescale="monthly",
                scenario=[scenario],
                time_slice=TIMESPAN,
                latitude=lat_bounds,
                longitude=lon_bounds
            )
            if d is not None and d.time.size > 0:
                d = preprocess_data(d, var_key)
                if scenario not in lazy_data:
                    lazy_data[scenario] = {}
                lazy_data[scenario][var_key] = d
                print(f"    {scenario:20s} | {var_key:6s}: OK ({d.shape}) {_time.perf_counter()-t0:.0f}s")
            else:
                print(f"    {scenario:20s} | {var_key:6s}: no data")
        except Exception as e:
            print(f"    {scenario:20s} | {var_key:6s}: ERROR {e}")
    
    fetch_time = _time.perf_counter() - fetch_start
    print(f"  Phase 1 done in {fetch_time:.0f}s")
    
    # Compute T_Avg for each scenario (still lazy — just adds dask graphs)
    for scenario in lazy_data:
        raw = lazy_data[scenario]
        if raw.get("T_Max") is not None and raw.get("T_Min") is not None:
            raw["T_Avg"] = (raw["T_Max"] + raw["T_Min"]) / 2
            raw["T_Avg"].attrs = raw["T_Max"].attrs
        else:
            raw["T_Avg"] = None

    # ================================================================
    # PHASE 2: Parallel .load() + processing
    #   Fire .load() for multiple (scenario, var) pairs concurrently.
    #   Each .load() pulls ~10MB from Cal-Adapt's Zarr store (~90s).
    #   Overlapping these is where the real speedup comes from.
    # ================================================================
    
    # Build list of units to process in parallel
    load_jobs = []
    for scenario in SCENARIOS:
        if scenario not in lazy_data:
            continue
        raw = lazy_data[scenario]
        for var_key in ["T_Avg", "Precip"]:
            if (region_name, scenario, var_key) in processed_units:
                continue
            if raw.get(var_key) is None:
                continue
            load_jobs.append((var_key, scenario, raw[var_key]))
    
    if not load_jobs:
        print(f"  No load jobs for {region_name}, skipping Phase 2")
        continue
    
    print(f"\n  Phase 2: Loading {len(load_jobs)} arrays in parallel threads...")
    load_start = _time.perf_counter()
    
    # Process results as they complete
    spatial_avgs = {}  # (scenario, var_key) -> spatial_avg DataArray
    with ThreadPoolExecutor(max_workers=min(len(load_jobs), 8)) as executor:
        futures = {}
        for var_key, scenario, lazy_da in load_jobs:
            f = executor.submit(_load_one, var_key, scenario, lazy_da, boundary_gdf)
            futures[f] = (var_key, scenario)
        
        done_count = 0
        for future in as_completed(futures):
            var_key, scenario = futures[future]
            result_vk, result_sc, result_avg = future.result()
            done_count += 1
            
            status = "OK" if result_avg is not None else "no data"
            print(f"    [{done_count}/{len(load_jobs)}] {scenario} | {var_key}: {status}")
            
            if result_avg is not None:
                spatial_avgs[(scenario, var_key)] = result_avg
    
    load_time = _time.perf_counter() - load_start
    print(f"  Phase 2 done in {load_time:.0f}s")

    # ================================================================
    # PHASE 3: Anomaly calculation + incremental CSV saves
    #   This is fast (pure in-memory pandas/xarray ops).
    # ================================================================
    
    print(f"\n  Phase 3: Anomaly calculation + CSV saves...")
    for scenario in SCENARIOS:
        for var_key in ["T_Avg", "Precip"]:
            if (region_name, scenario, var_key) in processed_units:
                continue
            if (scenario, var_key) not in spatial_avgs:
                continue
            
            elapsed = (_time.perf_counter() - run_start) / 60
            if completed_this_run > 0:
                avg_per = elapsed / completed_this_run
                eta = avg_per * (total_remaining - completed_this_run)
                eta_str = f"  ETA: ~{eta:.0f} min"
            else:
                eta_str = ""
            
            print(f"\n  [{completed_this_run + 1}/{total_remaining}] "
                  f"{region_name} | {scenario} | {var_key} "
                  f"({elapsed:.1f} min elapsed){eta_str}")
            
            spatial_avg = spatial_avgs[(scenario, var_key)]
            sdf = calculate_smoothed_anomalies(spatial_avg, var_key, scenario)
            
            if sdf is not None:
                sdf['Region'] = region_name
                cols_to_export = [c for c in required_cols if c in sdf.columns]
                export_df = sdf[cols_to_export].dropna(subset=['Anomaly'])
                
                # --- INCREMENTAL SAVE ---
                export_df.to_csv(output_filename, mode='a', header=False, index=False)
                
                print(f"    SAVED ({len(export_df)} rows)")
                processed_units.add((region_name, scenario, var_key))
                completed_this_run += 1
            else:
                print(f"    FAILED (no results)")
    
    # Free memory for this region
    del lazy_data, spatial_avgs
    gc.collect()

# --- Summary ---
total_time = _time.perf_counter() - run_start
print(f"\n{'=' * 60}")
print(f"Session complete.")
print(f"  Processed this run: {completed_this_run}/{total_remaining}")
print(f"  Total done:         {len(processed_units)}/{len(ALL_WORK)}")
print(f"  Session time:       {total_time/60:.1f} min")
print(f"  Output:             {output_filename}")
if len(processed_units) < len(ALL_WORK):
    still_left = len(ALL_WORK) - len(processed_units)
    print(f"\n  {still_left} units remaining — re-run this cell to continue!")
else:
    print(f"\n  ALL DONE!")
print(f"{'=' * 60}")